# P27 — Dominar el go con redes neuronales profundas y búsqueda en árbol

## 1. Título y paper

**Paper:** *Mastering the game of Go with deep neural networks and tree search*  
**Autoría:** David Silver, Aja Huang, Chris J. Maddison, y otros (DeepMind)  
**Año y venue:** 2016 · Nature 529, 484–489 (2016)  
**Nivel:** L4 · **Motor:** `alphago`  
**Ficha completa:** [`P27_alphago`](../../papers/foundational/P27_alphago/README.md)

**Hito:** Une las dos tradiciones de la IA: la búsqueda simbólica de la parte 01 y el aprendizaje profundo de la parte 04, en un solo sistema.

- [DOI (Nature 529, 484–489)](https://doi.org/10.1038/nature16961)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: El go tiene un espacio de estados y un factor de ramificación que hacen inviable la búsqueda exhaustiva, y no existía una función de evaluación de posiciones suficientemente buena.
2. Ejecutar una implementación mínima de la propuesta: Una red de políticas que propone jugadas plausibles y una red de valor que evalúa posiciones, usadas para guiar y truncar una búsqueda de Monte Carlo en árbol.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P26
- P04
- búsqueda en árbol y MCTS clásicos


## 4. Intuición

Un buen jugador no calcula todas las jugadas: su intuición descarta el 99 % y solo analiza a fondo las tres o cuatro que valen la pena. AlphaGo hace exactamente eso: una red da la intuición, la búsqueda hace el análisis.


## 5. Concepto mínimo

```text
red de políticas p(a|s)   → qué jugadas merecen considerarse (reduce la ANCHURA)
red de valor    v(s)      → cómo de buena es esta posición (reduce la PROFUNDIDAD)
búsqueda MCTS             → combina ambas, simula y decide
```

Sin el prior, la búsqueda se dispersa en un factor de ramificación inabordable. Sin la búsqueda, el prior propone pero **no verifica nada**.


## 6. Código explicado

El motor juega una posición de tres en raya con prior heurístico, con y sin búsqueda guiada.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('alphago', seed=7)['result']
print('posición:', r['posicion'])
print('solo política     → casilla', r['solo_politica'])
print('política+búsqueda → casilla', r['politica_mas_busqueda'])
show(r['valores_estimados_por_busqueda'])

## 7. Predicción antes de ejecutar

1. ¿Cuál es la respuesta correcta cuando el rival ocupa el centro?
2. ¿Qué información tiene la búsqueda que el prior no tiene?
3. ¿Por qué el prior sigue siendo necesario si la búsqueda evalúa?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('alphago', seed=semilla)['result']
    print(f"semilla {semilla:>2} · prior {r['solo_politica']} · búsqueda {r['politica_mas_busqueda']} "
          f"· ambas esquina: {r['ambas_eligen_esquina']}")

## 9. Salida interpretable

Ambas eligen esquina, pero solo la búsqueda produce **un número por casilla**. Esa es la diferencia operativa: una preferencia no se puede comparar ni auditar; una estimación de valor, sí. Y con más simulaciones, esa estimación mejora — el prior no mejora con nada.


## 10. Comentario pedagógico

AlphaGo es donde se juntan las dos tradiciones que el programa enseña por separado: la búsqueda simbólica de la parte 01 y el aprendizaje profundo de la parte 04. Ninguna de las dos habría bastado sola, y eso es lo que hay que llevarse.


## 11. Error o anti-patrón deliberado

Anti-patrón: presentarlo como «la red neuronal venció al campeón». La red sola no vence a nadie.


In [ ]:
print('Sin busqueda: la red propone la jugada mas plausible SEGUN PARTIDAS VISTAS.')
print('No comprueba si funciona en ESTA posicion concreta.')
print('El titulo del paper nombra las dos piezas: redes profundas Y busqueda en arbol.')

## 12. Corrección

La formulación correcta separa las tres contribuciones:


In [ ]:
contribuciones = {
    'red de políticas': 'reduce la anchura del árbol proponiendo jugadas plausibles',
    'red de valor': 'reduce la profundidad evaluando posiciones sin llegar al final',
    'MCTS': 'usa ambas para repartir un presupuesto de simulaciones y decidir',
    'autojuego': 'genera los datos con los que se refinan las redes',
}
show(contribuciones)

## 13. Desafío guiado

Reparte el presupuesto de simulaciones de forma uniforme en vez de según el prior y compara.


In [ ]:
libres = 8
for presupuesto in (8, 40, 200):
    print(f'{presupuesto:>3} simulaciones · uniforme: {presupuesto // libres} por jugada '
          f'· guiado: hasta {int(presupuesto * 0.25)} en la más prometedora')

## 14. Desafío autónomo

Implementa MCTS con UCT sobre tres en raya o conecta-4, con y sin prior heurístico. Mide la tasa de victoria frente a un oponente aleatorio en función del número de simulaciones, y localiza a partir de cuántas el prior deja de aportar.


## 15. Evidencia de aprendizaje

Guarda las jugadas elegidas por ambos métodos, los valores estimados por la búsqueda y tu explicación de qué reduce la anchura y qué reduce la profundidad.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P27_alphago/README.md) · evaluación formal: [`assessments/papers/P27_alphago.md`](../../assessments/papers/P27_alphago.md)


## 16. Cierre

Búsqueda y aprendizaje ya colaboran en un juego con reglas. Trasladar eso al **lenguaje**, donde no hay reglas ni marcador, exige otra idea.


## 17. Conexión con el siguiente hito

- P29
- P22

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
